### 1.Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

BASE_DIR = Path("/content/drive/MyDrive")
OUT_DIR = BASE_DIR / "Task 3 NLP"           # all Task 3 outputs
OUT_DIR.mkdir(exist_ok=True)

print("Output folder:", OUT_DIR, "| exists:", OUT_DIR.exists())

Output folder: /content/drive/MyDrive/Task 3 NLP | exists: True


In [ ]:
SELECT_METRIC = "ROC-AUC"
LR_C_GRID = [0.01, 0.1, 1, 10]
XGB_GRID = [(depth, lr) for depth in (2, 3, 4) for lr in (0.03, 0.05, 0.1)]

N_COMPONENTS = 30
KEEP_HAS_NEWS = True
USE_ONLY_USABLE = True

TRAIN_END = "2024-12-31"
VAL_END = "2026-03-31"
SEED = 42

### 2. Loading data and rebuilding the fused dataset

In [ ]:
financial = pd.read_csv(BASE_DIR / "financial_features.csv")
daily_nlp = pd.read_csv(BASE_DIR / "daily_nlp_features.csv")
articles = pd.read_csv(BASE_DIR / "article_features_aligned.csv")

financial["date"] = pd.to_datetime(financial["date"]).dt.normalize()
daily_nlp["trade_date"] = pd.to_datetime(daily_nlp["trade_date"]).dt.normalize()
articles["trade_date"] = pd.to_datetime(articles["trade_date"]).dt.normalize()

print("Financial:", financial.shape)
print("Daily NLP:", daily_nlp.shape)
print("Articles :", articles.shape)

Financial: (1181, 12)
Daily NLP: (1202, 67)
Articles : (3577, 67)


In [ ]:
FIN_COLS = [
    "fx_return_lag_0", "fx_return_lag_1", "fx_return_lag_2", "fx_return_lag_3",
    "fx_return_lag_4", "fx_volatility_5", "fx_volatility_20", "fx_momentum_5",
    "fx_day_of_week",
]
NLP_COLS = [col for col in daily_nlp.columns if col != "trade_date"]

missing = [col for col in FIN_COLS if col not in financial.columns]
assert not missing, f"Missing financial columns: {missing}"

print("Financial features:", len(FIN_COLS))
print("Task 2 NLP features:", len(NLP_COLS))

Financial features: 9
Task 2 NLP features: 66


In [ ]:
combined = (
    financial.rename(columns={"date": "trade_date"})
    .merge(daily_nlp, on="trade_date", how="left", validate="one_to_one")
    .sort_values("trade_date")
    .reset_index(drop=True)
)

assert len(combined) == len(financial), "Row count changed during the merge"
assert combined["trade_date"].is_unique, "Duplicate trading dates"
assert combined[FIN_COLS + NLP_COLS].isna().sum().sum() == 0, "Missing feature values"

print("Combined:", combined.shape)
print("Dates:", combined["trade_date"].min().date(), "to", combined["trade_date"].max().date())
print("News days:", int(combined["has_news"].sum()), "of", len(combined))

Combined: (1181, 78)
Dates: 2021-09-29 to 2026-08-31
News days: 385 of 1181


In [ ]:
def split_table(table):
    train = table[table["trade_date"] <= TRAIN_END]
    val = table[(table["trade_date"] > TRAIN_END) & (table["trade_date"] <= VAL_END)]
    test = table[table["trade_date"] > VAL_END]
    return {"train": train, "val": val, "test": test}


def to_y(part):
    return (part["target"] == "UP").astype(int)


parts = split_table(combined)

for name, part in parts.items():
    y = to_y(part)
    print(f"{name:5s} rows={len(part):4d}  UP={int(y.sum()):3d}  DOWN={int((1 - y).sum()):3d}  "
          f"news days={int(part['has_news'].sum()):3d}  "
          f"majority acc={max(y.mean(), 1 - y.mean()):.4f}")

train rows= 790  UP=444  DOWN=346  news days=252  majority acc=0.5620
val   rows= 292  UP=158  DOWN=134  news days=104  majority acc=0.5411
test  rows=  99  UP= 54  DOWN= 45  news days= 29  majority acc=0.5455


### 3. Evaluation functions

In [ ]:
def evaluate_model(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
    }


def majority_accuracy(y_true):
    share_up = np.mean(y_true)
    return max(share_up, 1 - share_up)


def bootstrap_accuracy_ci(y_true, y_pred, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    correct = (np.asarray(y_true) == np.asarray(y_pred)).astype(float)
    idx = rng.integers(0, len(correct), size=(n_boot, len(correct)))
    scores = correct[idx].mean(axis=1)
    return np.percentile(scores, 2.5), np.percentile(scores, 97.5)

### 4. Classifiers and tuning loop

In [ ]:
def make_model(kind, params):
    if kind == "LR":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(C=params["C"], max_iter=2000)),
        ])
    return XGBClassifier(
        n_estimators=200,
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        eval_metric="logloss",
    )


DEFAULT_PARAMS = {
    "LR": {"C": 1.0},
    "XGB": {"max_depth": 3, "learning_rate": 0.05},
}
PARAM_GRID = {
    "LR": [{"C": c} for c in LR_C_GRID],
    "XGB": [{"max_depth": d, "learning_rate": lr} for d, lr in XGB_GRID],
}

In [ ]:
def fit_and_score(table, feature_cols, kind, params):
    parts = split_table(table)
    model = make_model(kind, params)
    model.fit(parts["train"][feature_cols], to_y(parts["train"]))

    out = {"model": model}
    for name in ("val", "test"):
        X, y = parts[name][feature_cols], to_y(parts[name])
        pred = model.predict(X)
        prob = model.predict_proba(X)[:, 1]
        out[name] = {"metrics": evaluate_model(y, pred, prob), "pred": pred, "prob": prob}
    return out


def tune(table, feature_cols, kind):
    log = []
    for params in PARAM_GRID[kind]:
        res = fit_and_score(table, feature_cols, kind, params)
        log.append({**params, **{f"Val {k}": v for k, v in res["val"]["metrics"].items()}})
    log = pd.DataFrame(log)
    best_row = log[f"Val {SELECT_METRIC}"].idxmax()
    return PARAM_GRID[kind][best_row], log

In [ ]:
results = []
tuning_logs = []
store = {}


def run_feature_set(label, table, feature_cols):
    y_test = to_y(split_table(table)["test"])
    for kind in ("LR", "XGB"):
        best_params, log = tune(table, feature_cols, kind)
        tuning_logs.append(log.assign(Model=label, Classifier=kind))

        for setting, params in (("untuned", DEFAULT_PARAMS[kind]), ("tuned", best_params)):
            res = fit_and_score(table, feature_cols, kind, params)
            ci_low, ci_high = bootstrap_accuracy_ci(y_test, res["test"]["pred"])
            results.append({
                "Model": label, "Classifier": kind, "Setting": setting,
                "Features": len(feature_cols), "Params": str(params),
                **{f"Val {k}": v for k, v in res["val"]["metrics"].items()},
                **{f"Test {k}": v for k, v in res["test"]["metrics"].items()},
                "Test Acc CI low": ci_low, "Test Acc CI high": ci_high,
            })
            store[(label, kind, setting)] = {**res, "feature_cols": feature_cols, "table": table}
    print("Done:", label, "|", len(feature_cols), "features")

### 5. Model A and Model B

In [ ]:
run_feature_set("A: Financial only", combined, FIN_COLS)
run_feature_set("B: Financial + Task 2 NLP", combined, FIN_COLS + NLP_COLS)

pd.DataFrame(results)[["Model", "Classifier", "Setting", "Params",
                       "Val ROC-AUC", "Val MCC", "Test Accuracy", "Test MCC"]].round(4)

Done: A: Financial only | 9 features
Done: B: Financial + Task 2 NLP | 75 features


,Model,Classifier,Setting,Params,Val ROC-AUC,Val MCC,Test Accuracy,Test MCC
0,A: Financial only,LR,untuned,{'C': 1.0},0.5295,0.1145,0.4949,-0.0843
1,A: Financial only,LR,tuned,{'C': 1},0.5295,0.1145,0.4949,-0.0843
2,A: Financial only,XGB,untuned,"{'max_depth': 3, 'learning_rate': 0.05}",0.5157,0.0292,0.4343,-0.1789
3,A: Financial only,XGB,tuned,"{'max_depth': 4, 'learning_rate': 0.1}",0.5374,0.0291,0.5152,0.0000
4,B: Financial + Task 2 NLP,LR,untuned,{'C': 1.0},0.4923,0.0402,0.5253,0.0197
5,B: Financial + Task 2 NLP,LR,tuned,{'C': 0.01},0.5273,0.1128,0.5253,-0.0046
6,B: Financial + Task 2 NLP,XGB,untuned,"{'max_depth': 3, 'learning_rate': 0.05}",0.5222,0.0062,0.4848,-0.0670
7,B: Financial + Task 2 NLP,XGB,tuned,"{'max_depth': 2, 'learning_rate': 0.1}",0.5388,0.0227,0.4646,-0.1164


### 6. Fusion function for Model C

In [ ]:
def build_daily_embedding_table(article_emb, method, n_components=N_COMPONENTS):
    df = article_emb.copy()
    df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.normalize()
    if "is_usable" in df.columns:
        if USE_ONLY_USABLE:
            print(f"{method}: dropping {int((df['is_usable'] == 0).sum())} flagged articles")
            df = df[df["is_usable"] == 1]
        df = df.drop(columns="is_usable")
    emb_cols = [col for col in df.columns if col not in ("article_id", "trade_date")]

    assert df["article_id"].is_unique, "Duplicate article_id in the embedding file"
    assert df[emb_cols].isna().sum().sum() == 0, "Missing values in the embeddings"

    X = df[emb_cols].to_numpy(dtype=float)
    if X.shape[1] > n_components:
        is_train = (df["trade_date"] <= TRAIN_END).to_numpy()
        pca = PCA(n_components=n_components, random_state=SEED).fit(X[is_train])
        X = pca.transform(X)
        print(f"{method}: PCA {len(emb_cols)} -> {n_components} dims, fitted on "
              f"{int(is_train.sum())} training articles, "
              f"explained variance {pca.explained_variance_ratio_.sum():.3f}")
    else:
        print(f"{method}: {len(emb_cols)} dims, no PCA needed")

    feat_cols = [f"{method}_{i}" for i in range(X.shape[1])]
    daily = (
        pd.DataFrame(X, columns=feat_cols)
        .assign(trade_date=df["trade_date"].to_numpy())
        .groupby("trade_date")[feat_cols].mean()
        .reset_index()
    )
    daily["has_news"] = 1

    table = (
        combined[["trade_date", "usd_idr", "target"] + FIN_COLS]
        .merge(daily, on="trade_date", how="left", validate="one_to_one")
    )
    table[feat_cols + ["has_news"]] = table[feat_cols + ["has_news"]].fillna(0)
    table["has_news"] = table["has_news"].astype(int)

    assert len(table) == len(combined) and table["trade_date"].is_unique
    print(f"{method}: {len(table)} rows, {int(table['has_news'].sum())} news days, "
          f"{len(feat_cols)} embedding features")

    model_cols = FIN_COLS + feat_cols + (["has_news"] if KEEP_HAS_NEWS else [])
    return table, model_cols

In [ ]:
svd_cols = [col for col in articles.columns if col.startswith("body_svd_")]
standin_emb = articles[["article_id", "trade_date"] + svd_cols]

standin_table, standin_cols = build_daily_embedding_table(standin_emb, "svd_standin")

new_vals = standin_table[[f"svd_standin_{i}" for i in range(len(svd_cols))]].to_numpy()
old_vals = combined[svd_cols].to_numpy()
print("Matches Task 2 daily SVD features:", np.allclose(new_vals, old_vals))
print("has_news matches Task 2:", (standin_table["has_news"] == combined["has_news"]).all())

svd_standin: 30 dims, no PCA needed
svd_standin: 1181 rows, 385 news days, 30 embedding features
Matches Task 2 daily SVD features: True
has_news matches Task 2: True


### 7. Model C

In [ ]:
EMBEDDING_FILES = {
    "finbert": "finbert_pca30.csv",
    "roberta": "roberta_pca30.csv",
    "fasttext": "fasttext_pca30.csv",
    "mpnet": "mpnet_pca30.csv",
}

model_c_labels = []

label = "C-ref: Financial + TF-IDF/SVD (stand-in)"
run_feature_set(label, standin_table, standin_cols)

for method, file_name in EMBEDDING_FILES.items():
    emb = pd.read_csv(OUT_DIR / file_name)
    table, model_cols = build_daily_embedding_table(emb, method)
    label = f"C: Financial + {method}"
    run_feature_set(label, table, model_cols)
    model_c_labels.append(label)

print("Real Model C methods available:", model_c_labels if model_c_labels else "none yet")

Done: C-ref: Financial + TF-IDF/SVD (stand-in) | 40 features
finbert: dropping 49 flagged articles
finbert: 30 dims, no PCA needed
finbert: 1181 rows, 385 news days, 30 embedding features
Done: C: Financial + finbert | 40 features
roberta: dropping 49 flagged articles
roberta: 30 dims, no PCA needed
roberta: 1181 rows, 385 news days, 30 embedding features
Done: C: Financial + roberta | 40 features
fasttext: dropping 49 flagged articles
fasttext: 30 dims, no PCA needed
fasttext: 1181 rows, 385 news days, 30 embedding features
Done: C: Financial + fasttext | 40 features
mpnet: dropping 49 flagged articles
mpnet: 30 dims, no PCA needed
mpnet: 1181 rows, 385 news days, 30 embedding features
Done: C: Financial + mpnet | 40 features
Real Model C methods available: ['C: Financial + finbert', 'C: Financial + roberta', 'C: Financial + fasttext', 'C: Financial + mpnet']


### 8. Ablation Matrix

In [ ]:
ablation = pd.DataFrame(results)
tuning_log = pd.concat(tuning_logs, ignore_index=True)

y_val = to_y(parts["val"])
y_test = to_y(parts["test"])
print(f"Majority-class accuracy | validation: {majority_accuracy(y_val):.4f} | "
      f"test: {majority_accuracy(y_test):.4f}")

ablation.to_csv(OUT_DIR / "ablation_matrix.csv", index=False)
tuning_log.to_csv(OUT_DIR / "tuning_log.csv", index=False)

show_cols = ["Model", "Classifier", "Setting", "Features", "Params",
             f"Val {SELECT_METRIC}", "Test Accuracy", "Test Acc CI low", "Test Acc CI high",
             "Test Macro F1", "Test MCC", "Test ROC-AUC"]
ablation[show_cols].round(4)

Majority-class accuracy | validation: 0.5411 | test: 0.5455


,Model,Classifier,Setting,Features,Params,Val ROC-AUC,Test Accuracy,Test Acc CI low,Test Acc CI high,Test Macro F1,Test MCC,Test ROC-AUC
0,A: Financial only,LR,untuned,9,{'C': 1.0},0.5295,0.4949,0.3939,0.5859,0.4228,-0.0843,0.5004
1,A: Financial only,LR,tuned,9,{'C': 1},0.5295,0.4949,0.3939,0.5859,0.4228,-0.0843,0.5004
2,A: Financial only,XGB,untuned,9,"{'max_depth': 3, 'learning_rate': 0.05}",0.5157,0.4343,0.3434,0.5354,0.4021,-0.1789,0.4519
3,A: Financial only,XGB,tuned,9,"{'max_depth': 4, 'learning_rate': 0.1}",0.5374,0.5152,0.4141,0.6162,0.4923,0.0000,0.4975
4,B: Financial + Task 2 NLP,LR,untuned,75,{'C': 1.0},0.4923,0.5253,0.4343,0.6162,0.5006,0.0197,0.5259
5,B: Financial + Task 2 NLP,LR,tuned,75,{'C': 0.01},0.5273,0.5253,0.4242,0.6263,0.4618,-0.0046,0.4909
6,B: Financial + Task 2 NLP,XGB,untuned,75,"{'max_depth': 3, 'learning_rate': 0.05}",0.5222,0.4848,0.3939,0.5758,0.4581,-0.0670,0.4823
7,B: Financial + Task 2 NLP,XGB,tuned,75,"{'max_depth': 2, 'learning_rate': 0.1}",0.5388,0.4646,0.3636,0.5657,0.4312,-0.1164,0.4395
8,C-ref: Financial + TF-IDF/SVD (stand-in),LR,untuned,40,{'C': 1.0},0.5390,0.5354,0.4343,0.6263,0.5088,0.0398,0.5025
9,C-ref: Financial + TF-IDF/SVD (stand-in),LR,tuned,40,{'C': 0.01},0.5421,0.5556,0.4644,0.6566,0.4834,0.0685,0.5016


In [ ]:
tuned_c = ablation[(ablation["Setting"] == "tuned") & (ablation["Model"].isin(model_c_labels))]

if tuned_c.empty:
    best_c_label = None
    print("No real Model C embeddings yet. Add Person 1's files to EMBEDDING_FILES.")
else:
    best_c = tuned_c.loc[tuned_c[f"Val {SELECT_METRIC}"].idxmax()]
    best_c_label = best_c["Model"]
    print("Selected on validation:", best_c_label, "|", best_c["Classifier"],
          "|", f"Val {SELECT_METRIC} = {best_c[f'Val {SELECT_METRIC}']:.4f}")

Selected on validation: C: Financial + mpnet | LR | Val ROC-AUC = 0.5459


### 9. Feature Importance

In [ ]:
def feature_importance(label, kind):
    entry = store[(label, kind, "tuned")]
    model, cols = entry["model"], entry["feature_cols"]
    if kind == "LR":
        values = model.named_steps["model"].coef_[0]
        frame = pd.DataFrame({"feature": cols, "coefficient": values})
        frame["importance"] = frame["coefficient"].abs()
    else:
        frame = pd.DataFrame({"feature": cols, "importance": model.feature_importances_})
    frame = frame.sort_values("importance", ascending=False).reset_index(drop=True)
    frame.insert(0, "Classifier", kind)
    frame.insert(0, "Model", label)
    return frame


focus_label = best_c_label or "B: Financial + Task 2 NLP"
importance = pd.concat(
    [feature_importance(focus_label, kind) for kind in ("LR", "XGB")], ignore_index=True
)
importance.to_csv(OUT_DIR / "feature_importance.csv", index=False)

print("Model inspected:", focus_label)
for kind in ("LR", "XGB"):
    print(f"\nTop 10 features, {kind}:")
    print(importance[importance["Classifier"] == kind].head(10).drop(columns=["Model", "Classifier"])
          .dropna(axis=1, how="all").round(4).to_string(index=False))

Model inspected: C: Financial + mpnet

Top 10 features, LR:
        feature  coefficient  importance
fx_return_lag_1      -0.1149      0.1149
        mpnet_4      -0.0928      0.0928
        mpnet_0      -0.0830      0.0830
       mpnet_24      -0.0804      0.0804
fx_return_lag_3       0.0763      0.0763
       has_news       0.0646      0.0646
       mpnet_21      -0.0630      0.0630
fx_return_lag_0       0.0571      0.0571
       mpnet_27      -0.0567      0.0567
fx_volatility_5       0.0546      0.0546

Top 10 features, XGB:
 feature  importance
mpnet_15      0.0356
mpnet_17      0.0347
mpnet_29      0.0327
mpnet_28      0.0322
mpnet_24      0.0311
mpnet_25      0.0303
 mpnet_7      0.0286
mpnet_16      0.0285
mpnet_19      0.0283
 mpnet_5      0.0278


In [ ]:
def feature_group(name):
    if name in FIN_COLS:
        return "financial"
    if name in ("has_news", "news_count"):
        return "news availability"
    if name.startswith("body_svd_"):
        return "TF-IDF/SVD"
    if name.startswith(("title_", "body_lm_")):
        return "sentiment lexicons"
    if name.startswith(("ent_", "gdelt_")):
        return "entities / GDELT"
    return "embedding"


importance["group"] = importance["feature"].map(feature_group)
group_share = (
    importance.groupby(["Classifier", "group"])["importance"].sum()
    .groupby(level=0).transform(lambda s: s / s.sum())
    .unstack(0).round(3)
)
group_share

Classifier,LR,XGB
group,,
embedding,0.656,0.773
financial,0.302,0.220
news availability,0.043,0.006


### 10. Error analysis

In [ ]:
test_news_days = int(parts["test"]["has_news"].sum())
ERROR_SPLIT = "test" if test_news_days >= 30 else "val"
print("Test news days:", test_news_days, "-> error analysis on:", ERROR_SPLIT)

Test news days: 29 -> error analysis on: val


In [ ]:
part = parts[ERROR_SPLIT]
y_true = to_y(part).to_numpy()
is_news = (part["has_news"] == 1).to_numpy()

rows = []
for (label, kind, setting), entry in store.items():
    if setting != "tuned":
        continue
    correct = entry[ERROR_SPLIT]["pred"] == y_true
    rows.append({
        "Model": label, "Classifier": kind,
        "News days": int(is_news.sum()), "Acc news days": correct[is_news].mean(),
        "No-news days": int((~is_news).sum()), "Acc no-news days": correct[~is_news].mean(),
    })

news_comparison = pd.DataFrame(rows)
news_comparison.to_csv(OUT_DIR / f"news_vs_no_news_{ERROR_SPLIT}.csv", index=False)
news_comparison.round(4)

,Model,Classifier,News days,Acc news days,No-news days,Acc no-news days
0,A: Financial only,LR,104,0.5962,188,0.5532
1,A: Financial only,XGB,104,0.5385,188,0.5106
2,B: Financial + Task 2 NLP,LR,104,0.5577,188,0.5745
3,B: Financial + Task 2 NLP,XGB,104,0.5096,188,0.5266
4,C-ref: Financial + TF-IDF/SVD (stand-in),LR,104,0.6058,188,0.5798
5,C-ref: Financial + TF-IDF/SVD (stand-in),XGB,104,0.5385,188,0.5319
6,C: Financial + finbert,LR,104,0.5192,188,0.5691
7,C: Financial + finbert,XGB,104,0.5385,188,0.5160
8,C: Financial + roberta,LR,104,0.5962,188,0.5691
9,C: Financial + roberta,XGB,104,0.5096,188,0.5213


In [ ]:
KIND = "LR"
pred_a = store[("A: Financial only", KIND, "tuned")][ERROR_SPLIT]["pred"]
pred_text = store[(focus_label, KIND, "tuned")][ERROR_SPLIT]["pred"]

by_date = pd.DataFrame({
    "trade_date": part["trade_date"].to_numpy(),
    "actual": part["target"].to_numpy(),
    "has_news": part["has_news"].to_numpy(),
    "A_correct": pred_a == y_true,
    "text_correct": pred_text == y_true,
})
by_date["outcome"] = "same result"
by_date.loc[~by_date["A_correct"] & by_date["text_correct"], "outcome"] = "text helped"
by_date.loc[by_date["A_correct"] & ~by_date["text_correct"], "outcome"] = "text added noise"

by_date.to_csv(OUT_DIR / f"error_analysis_by_date_{ERROR_SPLIT}.csv", index=False)

print("Compared:", focus_label, "vs Model A |", KIND, "|", ERROR_SPLIT)
pd.crosstab(by_date["outcome"], by_date["has_news"].map({1: "news day", 0: "no-news day"}))

Compared: C: Financial + mpnet vs Model A | LR | val


has_news,news day,no-news day
outcome,,
same result,81,175
text added noise,14,4
text helped,9,9


In [ ]:
by_date["month"] = pd.to_datetime(by_date["trade_date"]).dt.to_period("M")
monthly = by_date.groupby("month").agg(
    days=("actual", "size"),
    news_days=("has_news", "sum"),
    acc_model_A=("A_correct", "mean"),
    acc_text_model=("text_correct", "mean"),
).round(3)
monthly.to_csv(OUT_DIR / f"monthly_accuracy_{ERROR_SPLIT}.csv")
monthly

,days,news_days,acc_model_A,acc_text_model
month,,,,
2025-01,19,10,0.421,0.579
2025-02,20,11,0.600,0.600
2025-03,19,1,0.632,0.526
2025-04,16,4,0.625,0.625
2025-05,17,5,0.529,0.529
2025-06,18,7,0.444,0.389
2025-07,23,12,0.609,0.609
2025-08,20,2,0.450,0.550
2025-09,21,7,0.571,0.667


### 11. Saved Outputs

In [ ]:
combined.to_csv(OUT_DIR / "combined_dataset_task3.csv", index=False)

print("Files in", OUT_DIR)
for path in sorted(OUT_DIR.glob("*.csv")):
    print(" -", path.name)

Files in /content/drive/MyDrive/Task 3 NLP
 - ablation_matrix.csv
 - combined_dataset_task3.csv
 - error_analysis_by_date_val.csv
 - fasttext_pca30.csv
 - feature_importance.csv
 - finbert_pca30.csv
 - monthly_accuracy_val.csv
 - mpnet_pca30.csv
 - news_vs_no_news_val.csv
 - roberta_pca30.csv
 - tuning_log.csv
